In [1]:
"""
This script gets data from ddl on multiple cores and generates a netcdf file per location and station.

"""

import ddlpy
import datetime as dt
import matplotlib.pyplot as plt
plt.close("all")
import xarray as xr
import os
from concurrent.futures import ProcessPoolExecutor
import glob

# enabling debug logging so we can see what happens in the background
import logging
logging.basicConfig()
logging.getLogger("ddlpy").setLevel(logging.DEBUG)

In [2]:
def get_data(location, start_date, end_date, dir_output, overwrite=True):
    station_id = location.name
    station_messageid = location["Locatie_MessageID"]
    grootheid_code = location["Grootheid.Code"]
    filename = os.path.join(dir_output, f"{station_id}-{station_messageid}-{grootheid_code}.nc")

    if os.path.isfile(filename) and overwrite is False:
        print("{station_id}: netcdf file already exists and overwrite=False, skipping")
        return

    measurements = ddlpy.measurements(location, start_date=start_date, end_date=end_date)

    if measurements.empty:
        print(f"{station_id}: no measurements found")
        return

    print(f"{station_id}: writing retrieved data to netcdf file")

    # convert to xarray: constant columns are converted to attributes to save disk space
    # except the columns in always_preserve
    always_preserve = [
        "WaarnemingMetadata.Statuswaarde",
        "WaarnemingMetadata.Kwaliteitswaardecode",
        "WaardeBepalingsMethode.Code",
        "Meetwaarde.Waarde_Numeriek",
    ]
    ds = ddlpy.dataframe_to_xarray(measurements, always_preserve=always_preserve)

    # write to netcdf file. NETCDF3_CLASSIC or NETCDF4_CLASSIC automatically converts
    # variables of dtype <U to |S which saves a lot of disk space
    ds.to_netcdf(filename, format="NETCDF4_CLASSIC")
    return filename

In [10]:
dir_output = "./ddl_retrieved_data"
os.makedirs(dir_output, exist_ok=True)

# get locations
locations = ddlpy.locations()
bool_stations = locations.index.isin(["ijmuiden.buitenhaven", "dantziggat.zuid", "hoekvanholland", "ameland.nes"])
bool_stations = locations.index.isin(["ijmuiden.buitenhaven"])
bool_stations = locations.index.isin(['hollandsekust.zuid.windparknoord', 'hollandsekust.noord.windparknoord'])

bool_procestype = locations["ProcesType"].isin(["meting"])  # meting/astronomisch/verwachting
bool_grootheid = locations["Grootheid.Code"].isin(["WATHTE"])  # waterlevel (WATHTE)
bool_groepering = locations["Groepering.Code"].isin([""])  # timeseries ("") versus extremes (GETETM2/GETETMSL2/GETETBRKD2/GETETBRKDMSL2)
bool_hoedanigheid = locations["Hoedanigheid.Code"].isin(["NAP"])  # vertical reference (NAP/MSL)
selected = locations.loc[
    bool_stations
    & bool_procestype
    & bool_groepering
]

start_date = dt.datetime(2024, 12, 1)
end_date = dt.datetime(2025, 3, 1)

# normal code
for station_code, location in selected.iterrows():
    get_data(location, start_date, end_date, dir_output)
# parallel code
# with ProcessPoolExecutor(max_workers=3) as executor:
#     for station_code, location in selected.iterrows():
#         executor.submit(get_data, location, start_date, end_date, dir_output)

INFO:ddlpy.ddlpy:Loading Waterwebservices catalog from cache
100%|██████████| 3/3 [00:00<00:00,  5.83it/s]
DEBUG:ddlpy.ddlpy:no data found for this station and time extent


hollandsekust.zuid.windparknoord: no measurements found


100%|██████████| 3/3 [00:00<00:00,  9.46it/s]
DEBUG:ddlpy.ddlpy:no data found for this station and time extent


hollandsekust.zuid.windparknoord: no measurements found


100%|██████████| 3/3 [00:00<00:00,  8.79it/s]
DEBUG:ddlpy.ddlpy:no data found for this station and time extent


...

hollandsekust.zuid.windparknoord: no measurements found


100%|██████████| 3/3 [00:00<00:00,  9.26it/s]
DEBUG:ddlpy.ddlpy:no data found for this station and time extent


hollandsekust.noord.windparknoord: no measurements found


 33%|███▎      | 1/3 [00:00<00:00,  9.62it/s]DEBUG:ddlpy.ddlpy:Requesting at https://ddapi20-waterwebservices.rijkswaterstaat.nl/ONLINEWAARNEMINGENSERVICES/OphalenWaarnemingen with request: {"AquoPlusWaarnemingMetadata": {"AquoMetadata": {"Compartiment": {"Code": "OW"}, "Grootheid": {"Code": "Th3"}, "Eenheid": {"Code": "graad"}, "Hoedanigheid": {"Code": "WARNDN"}, "Parameter": {"Code": "NVT"}, "BioTaxon": {"Code": "NVT"}, "Orgaan": {"Code": "NVT"}, "Groepering": {"Code": ""}, "Typering": {"Code": "NVT"}, "WaardeBewerkingsMethode": {"Code": "NVT"}, "ProcesType": "meting"}}, "Locatie": {"Code": "hollandsekust.noord.windparknoord"}, "Periode": {"Begindatumtijd": "2025-01-01T00:00:00.000+00:00", "Einddatumtijd": "2025-02-01T00:00:00.000+00:00"}}
DEBUG:ddlpy.ddlpy:Requesting at https://ddapi20-waterwebservices.rijkswaterstaat.nl/ONLINEWAARNEMINGENSERVICES/OphalenWaarnemingen with request: {"AquoPlusWaarnemingMetadata": {"AquoMetadata": {"Compartiment": {"Code": "OW"}, "Grootheid": {"Code": 

hollandsekust.noord.windparknoord: no measurements found


In [11]:
selected

,Locatie_MessageID,Lat,Lon,Coordinatenstelsel,Naam,Omschrijving,Parameter_Wat_Omschrijving,ProcesType,Compartiment.Code,Compartiment.Omschrijving,...,BioTaxon.Code,BioTaxon.Omschrijving,Orgaan.Code,Orgaan.Omschrijving,Groepering.Code,Groepering.Omschrijving,Typering.Code,Typering.Omschrijving,WaardeBewerkingsMethode.Code,WaardeBewerkingsMethode.Omschrijving
Code,,,,,,,,,,,,,,,,,,,,,
hollandsekust.zuid.windparknoord,8152,52.393,4.056,ETRS89,"Hollandse Kust, Zuid Windpark Noord","Hollandse Kust, Zuid Windpark Noord",Golffrequentie bij maximum v.h. variantiedicht...,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT
hollandsekust.zuid.windparknoord,8152,52.393,4.056,ETRS89,"Hollandse Kust, Zuid Windpark Noord","Hollandse Kust, Zuid Windpark Noord",Gem. richting deining tov ware noorden in spec...,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT
hollandsekust.zuid.windparknoord,8152,52.393,4.056,ETRS89,"Hollandse Kust, Zuid Windpark Noord","Hollandse Kust, Zuid Windpark Noord",Temperatuur in Oppervlaktewater in oC,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT
hollandsekust.zuid.windparknoord,8152,52.393,4.056,ETRS89,"Hollandse Kust, Zuid Windpark Noord","Hollandse Kust, Zuid Windpark Noord",Stroomrichting in Oppervlaktewater t.o.v. ware...,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT
hollandsekust.zuid.windparknoord,8152,52.393,4.056,ETRS89,"Hollandse Kust, Zuid Windpark Noord","Hollandse Kust, Zuid Windpark Noord",Significante golfhoogte in het spectrale domei...,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT
hollandsekust.noord.windparknoord,3946,52.783,4.285,ETRS89,"Hollandse Kust, Noord Windpark Noord","Hollandse Kust, Noord Windpark Noord",Temperatuur in Oppervlaktewater in oC,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT
hollandsekust.noord.windparknoord,3946,52.783,4.285,ETRS89,"Hollandse Kust, Noord Windpark Noord","Hollandse Kust, Noord Windpark Noord",Gem. richting deining tov ware noorden in spec...,meting,OW,Oppervlaktewater,...,NVT,NVT,NVT,NVT,,,NVT,NVT,NVT,NVT


In [15]:
measurements = ddlpy.measurements(selected.iloc[0], "2024-12-01", "2025-03-01")

100%|██████████| 3/3 [00:01<00:00,  1.58it/s]
DEBUG:ddlpy.ddlpy:2 duplicated values dropped


In [ ]:
measurements